# VISWORD on Colab — parallel experiment runner

Project: <https://github.com/hkanpak21/Comp447_VISWORD>

## How to use

1. **Set GPU runtime** (Runtime → Change runtime type → T4 / A100).
2. Run cells **1–5** once per session — repo clone, dependencies, HF token, Drive mount.
3. In cell **6**, set `EXPERIMENT` to one of:
   - `'zeroshot_grid'` — encode 6 image + 4 text encoders on a 2 000-page eval slice, compute Protocol-A retrieval per encoder + the Platonic alignment grid (~2.5h on T4, faster on A100). **No training.**
   - `'finetune_dinov2_salad'` — fine-tune DINOv2 + SALAD aggregator on 30 k pages × 3 ep (~3-4h on T4, ~1.5h on A100).
   - `'finetune_clip_salad'` — fine-tune CLIP-ViT-B/16 + SALAD with `lr_bb=1e-7` on 30 k pages × 3 ep.
   - `'finetune_dinov2_mlp'` — fine-tune DINOv2 + MLP head.
   - `'finetune_clip_mlp'` — fine-tune CLIP + MLP head.
4. Run cells **7–10**. Results land in `/content/drive/MyDrive/visword_results/<EXPERIMENT>/` and you can share them back via the Drive folder or download.

## To run experiments in parallel

Open this notebook in **multiple Colab tabs** (different sessions) and set a different `EXPERIMENT` in each. They all write to the same Drive folder under different sub-paths so nothing collides. With 4 parallel Colab Pro sessions you can finish the full encoder-fine-tune grid in ~2h instead of ~12h on the cluster.

## 1. Sanity check — GPU and disk

In [ ]:
!nvidia-smi | head -20
!df -h /content | tail -1

## 2. Clone the project repo

In [ ]:
%%bash
set -e
if [ ! -d /content/visword/.git ]; then
  git clone --depth 50 https://github.com/hkanpak21/Comp447_VISWORD.git /content/visword
else
  cd /content/visword && git pull --ff-only
fi
cd /content/visword && git log -1 --oneline

### 2b. (Only if needed) BUNDLE FALLBACK

Run this cell **only** if the cluster's local commits aren't yet on GitHub `main` (see `notebooks/README.md`, Option C). Upload the cluster's `share/visword_local.bundle` to `MyDrive/visword_bundles/visword_local.bundle` first. If GitHub `main` already has the latest code, **skip** this cell.

In [ ]:
# BUNDLE FALLBACK — uncomment if GitHub main is behind the cluster's master.
import os, subprocess
BUNDLE_PATH = '/content/drive/MyDrive/visword_bundles/visword_local.bundle'
if False:    # set to True only if you're using Option C
    assert os.path.exists(BUNDLE_PATH), f'upload bundle to {BUNDLE_PATH} first'
    subprocess.run(['git', 'fetch', BUNDLE_PATH,
                    'refs/heads/master:cluster-master'],
                   cwd='/content/visword', check=True)
    subprocess.run(['git', 'reset', '--hard', 'cluster-master'],
                   cwd='/content/visword', check=True)
    print('cluster commits applied; HEAD now =',
          subprocess.check_output(['git', 'log', '-1', '--oneline'],
                                  cwd='/content/visword', text=True).strip())

## 3. Install dependencies

Colab images come with `torch` / `transformers`; we top up the rest. ~1 minute.

In [ ]:
%pip install -q open_clip_torch timm sentence-transformers \
    huggingface_hub imagehash sentencepiece protobuf pyyaml pydantic numpy pillow
import importlib, os, subprocess, sys
for m in ('torch','open_clip','timm','transformers','huggingface_hub','sentence_transformers'):
    try: importlib.import_module(m); print(f'OK {m}')
    except Exception as e: print(f'FAIL {m}: {e}')

# Vendor the SALAD repo (third_party/salad/ is gitignored — must be
# re-cloned per machine). Required by every fine-tune that uses the
# SALAD aggregator and by the model-factory provenance check.
if not os.path.isdir('/content/visword/third_party/salad'):
    print('vendoring SALAD...')
    res = subprocess.run(['bash', 'scripts/vendor_salad.sh'],
                         cwd='/content/visword',
                         capture_output=True, text=True)
    print(res.stdout)
    if res.returncode != 0:
        print('--- vendor_salad.sh stderr ---')
        print(res.stderr)
        raise RuntimeError(f'vendor_salad.sh exited {res.returncode}')
print('SALAD vendored:', os.path.isdir('/content/visword/third_party/salad'))

## 4. Hugging Face token + Google Drive

**HF token** (only needed for downloading wiki-ss; not needed if data is already cached):
1. Go to <https://huggingface.co/settings/tokens> and create a read-token.
2. In Colab, click the key icon (left sidebar) → **Add new secret** → name `HF_TOKEN` → paste the token → enable for this notebook.

**Drive** is mounted so all results land in `MyDrive/visword_results/`.

In [ ]:
import os
from google.colab import drive, userdata

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab secrets.')
except Exception as e:
    print(f'WARNING: HF_TOKEN not set ({e}). Add via the key icon (left sidebar).')

drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/visword_results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('results will be saved to:', RESULTS_DIR)

## 5. Pick the experiment

**Change this single variable** to control what runs in this session:

In [ ]:
# ============================================================
# EXPERIMENT — set me!
# ============================================================
EXPERIMENT = 'zeroshot_grid'

# Other valid values (uncomment / replace):
# EXPERIMENT = 'finetune_dinov2_salad'
# EXPERIMENT = 'finetune_dinov2_mlp'
# EXPERIMENT = 'finetune_clip_salad'
# EXPERIMENT = 'finetune_clip_mlp'

# Per-experiment settings — tune as needed.
NUM_EVAL_PAGES   = 2000   # eval pool for Protocol-A and Platonic
NUM_TRAIN_PAGES  = 30000  # only used for fine-tune experiments
EVAL_SEED        = 42

assert EXPERIMENT in {'zeroshot_grid', 'finetune_dinov2_salad', 'finetune_dinov2_mlp',
                      'finetune_clip_salad', 'finetune_clip_mlp'}, f'unknown {EXPERIMENT!r}'
print(f'experiment = {EXPERIMENT}')

## 6. Download the wiki-ss data slice we need

- For `zeroshot_grid`: 5 000 pages is plenty (the eval slice is 2 000; a few more give buffer for the random permutation).
- For fine-tune experiments: `NUM_TRAIN_PAGES + NUM_EVAL_PAGES` ~= 32 000 pages.

Downloads go to `/content/visword_data/wiki_ss/` and only run if not already cached. ~10 min for 5 k, ~50 min for 32 k.

In [ ]:
import os, json, shutil, subprocess
from pathlib import Path

DATA_ROOT = Path('/content/visword_data')
CACHE_DIR = DATA_ROOT / 'wiki_ss'
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if EXPERIMENT == 'zeroshot_grid':
    target_rows = max(NUM_EVAL_PAGES + 200, 3000)
else:
    target_rows = NUM_TRAIN_PAGES + NUM_EVAL_PAGES + 200

needs_download = True
manifest = CACHE_DIR / 'manifest.json'
if manifest.exists():
    have = json.loads(manifest.read_text()).get('num_rows', 0)
    if have >= target_rows:
        print(f'cache already has {have} rows (need {target_rows}); skip download.')
        needs_download = False
    else:
        print(f'cache has {have} rows; need {target_rows} more.')

if needs_download:
    print(f'downloading {target_rows} rows of Tevatron/wiki-ss-corpus...')
    cmd = [
        'python', '-m', 'visword.data.prefetch',
        '--data-dir', str(DATA_ROOT),
        '--dataset', 'wiki-ss',
        '--target-rows', str(target_rows),
    ]
    env = os.environ.copy()
    env['PYTHONPATH'] = '/content/visword/src'
    res = subprocess.run(cmd, env=env, cwd='/content/visword',
                         capture_output=True, text=True)
    print('--- stdout (tail) ---')
    print('\n'.join(res.stdout.splitlines()[-10:]))
    if res.returncode != 0:
        print('--- stderr (tail) ---')
        print('\n'.join(res.stderr.splitlines()[-15:]))
        raise RuntimeError(f'prefetch exited {res.returncode}')

print('cache:', json.loads(manifest.read_text()).get('num_rows', 0), 'rows')
print('disk:', subprocess.check_output(['du', '-sh', str(CACHE_DIR)]).decode().strip())

## 7. Run the experiment

This is the only long-running cell. Tail the output to monitor progress.

In [ ]:
import os, sys, time, subprocess, json
from pathlib import Path

VISWORD_ROOT = Path('/content/visword')
OUT_DIR = Path(RESULTS_DIR) / EXPERIMENT
OUT_DIR.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env['PYTHONPATH'] = str(VISWORD_ROOT / 'src')
env['HF_HOME'] = '/content/hf_cache'  # avoid /tmp filling up
Path('/content/hf_cache').mkdir(exist_ok=True)

# ---- self-healing prerequisites (idempotent) ---------------------
# (1) Pull latest code so a Drive-saved stale notebook still gets
#     up-to-date configs / scripts / src files at runtime.
subprocess.run(['git', 'pull', '--ff-only'], cwd=str(VISWORD_ROOT))
# (2) Vendor SALAD if the gitignored third_party/salad/ is missing.
#     Required by every fine-tune (provenance check imports salad_bridge).
if not (VISWORD_ROOT / 'third_party' / 'salad').is_dir():
    print('vendoring SALAD into third_party/salad/ ...')
    rc = subprocess.run(['bash', 'scripts/vendor_salad.sh'],
                        cwd=str(VISWORD_ROOT)).returncode
    assert rc == 0, f'vendor_salad.sh exited {rc}'
    print('  done')
assert (VISWORD_ROOT / 'third_party' / 'salad').is_dir(), \
    'SALAD vendor failed; run !cat /content/visword/scripts/vendor_salad.sh'

def stream(cmd, log_path):
    """Run cmd, tee stdout/stderr to log_path AND notebook output."""
    print(f'$ {" ".join(cmd)}', flush=True)
    with open(log_path, 'w') as logf:
        proc = subprocess.Popen(cmd, env=env, cwd=str(VISWORD_ROOT),
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='', flush=True); logf.write(line); logf.flush()
        proc.wait()
    return proc.returncode

t0 = time.time()

if EXPERIMENT == 'zeroshot_grid':
    # 6 zero-shot encoders × Protocol-A + Platonic alignment.
    # Each encoder: ~15-30 min on T4, ~5-10 min on A100.
    encoders = ['dinov2_cls', 'clip_image', 'siglip_image',
                'imagenet_vit', 'plain_vit', 'ijepa']
    for enc in encoders:
        out = OUT_DIR / f'{enc}_protocolA_n{NUM_EVAL_PAGES}.json'
        if out.exists():
            print(f'skip {enc} (already done at {out})'); continue
        print(f'\n=== zero-shot Protocol-A: {enc} ==='); sys.stdout.flush()
        rc = stream([
            'python', '-m', 'scripts.zeroshot_protocol_a',
            '--cache-dir', str(CACHE_DIR),
            '--encoder', enc,
            '--num-pages', str(NUM_EVAL_PAGES),
            '--seed', str(EVAL_SEED),
            '--out', str(out),
        ], log_path=OUT_DIR / f'{enc}.log')
        if rc != 0:
            print(f'  FAILED rc={rc}; continuing with the next encoder')
    # Platonic alignment grid
    print(f'\n=== Platonic alignment grid (n={NUM_EVAL_PAGES}) ==='); sys.stdout.flush()
    stream([
        'python', '-m', 'visword.analysis.platonic_alignment',
        '--cache-dir', str(CACHE_DIR),
        '--n-samples', str(NUM_EVAL_PAGES),
    ], log_path=OUT_DIR / 'platonic.log')

elif EXPERIMENT.startswith('finetune_'):
    cfg_map = {
        'finetune_dinov2_salad': 'configs/grid_dinov2_salad_30k.yaml',
        'finetune_dinov2_mlp':   'configs/grid_dinov2_mlp_30k.yaml',
        'finetune_clip_salad':   'configs/grid_clip_salad_30k.yaml',
        'finetune_clip_mlp':     'configs/grid_clip_mlp_30k.yaml',
    }
    cfg = cfg_map[EXPERIMENT]
    runs_root = '/content/visword_runs'
    Path(runs_root).mkdir(exist_ok=True)
    print(f'training {cfg} (data: {NUM_TRAIN_PAGES} pages × 3 ep)')
    rc = stream([
        'python', '-m', 'visword.train',
        '--config', cfg,
        '--runs-root', runs_root,
        '--set', f'data.wiki_ss_cache_dir={CACHE_DIR}',
        '--set', f'data.num_train_samples={NUM_TRAIN_PAGES}',
        '--set', f'data.num_eval_samples={NUM_EVAL_PAGES}',
    ], log_path=OUT_DIR / 'train.log')
    if rc != 0:
        print(f'training failed rc={rc}'); raise SystemExit(rc)
    # Find the run dir produced by train.py and run the new evals.
    run_dirs = sorted([d for d in Path(runs_root).iterdir() if d.is_dir()],
                      key=lambda d: d.stat().st_mtime, reverse=True)
    if not run_dirs:
        raise RuntimeError('no run dir under runs_root')
    RUN_DIR = run_dirs[0]
    print(f'\n=== eval_phase1_holdout (Protocol A) ===')
    stream(['python', '-m', 'visword.eval_phase1_holdout',
            '--run-dir', str(RUN_DIR)],
           log_path=OUT_DIR / 'eval_protocolA.log')
    print(f'\n=== eval_phase2_titleblanked-15% ===')
    stream(['python', '-m', 'visword.eval_phase2_titleblanked',
            '--run-dir', str(RUN_DIR), '--blank-top-frac', '0.15'],
           log_path=OUT_DIR / 'eval_titleblank15.log')
    # Copy run dir contents (configs, metrics, json results, ckpts)
    # into Drive so we can pick them up later.
    print(f'\ncopying run dir to Drive...')
    import shutil
    dest = OUT_DIR / 'run'
    if dest.exists(): shutil.rmtree(dest)
    shutil.copytree(RUN_DIR, dest, ignore=shutil.ignore_patterns('*.pt'))
    # Save the small best-checkpoint separately (compressed) — useful for further analysis.
    best = RUN_DIR / 'checkpoints' / 'best_phase1.pt'
    if best.exists():
        shutil.copy2(best, OUT_DIR / 'best_phase1.pt')
        print(f'  best_phase1.pt copied ({best.stat().st_size/1e6:.1f} MB)')

print(f'\n--- DONE in {(time.time()-t0)/60:.1f} min ---')
print(f'Results in: {OUT_DIR}')

## 8. Verify outputs

In [ ]:
from pathlib import Path
import json

for p in sorted(Path(OUT_DIR).rglob('*.json'))[:20]:
    rel = p.relative_to(OUT_DIR)
    print(f'\n--- {rel} ---')
    try:
        d = json.loads(p.read_text())
        # Pretty-print just the recall + sanity if present, else short summary.
        if 'recall' in d:
            print('  recall:', d['recall'])
        if 'sanity' in d:
            print('  sanity:', d.get('sanity'))
        if 'pairs' in d:
            print(f'  alignment pairs: {len(d["pairs"])} (sample below)')
            for k, v in list(d['pairs'].items())[:3]:
                print(f'    {k}: knn@10={v.get("mutual_knn_10"):.3f}')
    except Exception as e:
        print(f'  read error: {e}')

## 9. Done — share the Drive folder back

All outputs are in `MyDrive/visword_results/<EXPERIMENT>/`. To return them:

- **Easiest**: right-click the folder in Drive → *Share* → set link to anyone-with-link → paste the URL in chat.
- **Direct download**: in Colab, run the cell below to zip + download.

In [ ]:
# Optional: zip the results dir and download to laptop
import shutil, datetime
from google.colab import files
stamp = datetime.datetime.now().strftime('%Y-%m-%d_%H%M%S')
zip_path = f'/content/{EXPERIMENT}_{stamp}.zip'
shutil.make_archive(zip_path[:-4], 'zip', OUT_DIR)
print(f'zipped to {zip_path}, size {Path(zip_path).stat().st_size/1e6:.1f} MB')
files.download(zip_path)